In [1]:
import sqlite3
import os

# Create database file in the same folder
conn = sqlite3.connect("tennis.db")
cursor = conn.cursor()

print("✅ Connected to tennis.db")

# ---- TABLE 1: Categories ----
cursor.execute("""
CREATE TABLE IF NOT EXISTS categories (
    category_id   VARCHAR(50) PRIMARY KEY,
    category_name VARCHAR(100) NOT NULL
)
""")

# ---- TABLE 2: Competitions ----
cursor.execute("""
CREATE TABLE IF NOT EXISTS competitions (
    competition_id   VARCHAR(50) PRIMARY KEY,
    competition_name VARCHAR(100) NOT NULL,
    parent_id        VARCHAR(50),
    type             VARCHAR(20) NOT NULL,
    gender           VARCHAR(10) NOT NULL,
    category_id      VARCHAR(50),
    FOREIGN KEY (category_id) REFERENCES categories(category_id)
)
""")

# ---- TABLE 3: Complexes ----
cursor.execute("""
CREATE TABLE IF NOT EXISTS complexes (
    complex_id   VARCHAR(50) PRIMARY KEY,
    complex_name VARCHAR(100) NOT NULL
)
""")

# ---- TABLE 4: Venues ----
cursor.execute("""
CREATE TABLE IF NOT EXISTS venues (
    venue_id     VARCHAR(50) PRIMARY KEY,
    venue_name   VARCHAR(100) NOT NULL,
    city_name    VARCHAR(100) NOT NULL,
    country_name VARCHAR(100) NOT NULL,
    country_code CHAR(3) NOT NULL,
    timezone     VARCHAR(100) NOT NULL,
    complex_id   VARCHAR(50),
    FOREIGN KEY (complex_id) REFERENCES complexes(complex_id)
)
""")

# ---- TABLE 5: Competitors ----
cursor.execute("""
CREATE TABLE IF NOT EXISTS competitors (
    competitor_id VARCHAR(50) PRIMARY KEY,
    name          VARCHAR(100) NOT NULL,
    country       VARCHAR(100) NOT NULL,
    country_code  CHAR(3) NOT NULL,
    abbreviation  VARCHAR(10) NOT NULL
)
""")

# ---- TABLE 6: Competitor Rankings ----
cursor.execute("""
CREATE TABLE IF NOT EXISTS competitor_rankings (
    rank_id              INTEGER PRIMARY KEY AUTOINCREMENT,
    rank                 INT NOT NULL,
    movement             INT NOT NULL,
    points               INT NOT NULL,
    competitions_played  INT NOT NULL,
    competitor_id        VARCHAR(50),
    FOREIGN KEY (competitor_id) REFERENCES competitors(competitor_id)
)
""")

conn.commit()
print("✅ All 6 tables created successfully!")

# Show all tables
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
tables = cursor.fetchall()
print(f"\n📋 Tables in tennis.db: {[t[0] for t in tables]}")

✅ Connected to tennis.db
✅ All 6 tables created successfully!

📋 Tables in tennis.db: ['categories', 'competitions', 'complexes', 'venues', 'competitors', 'competitor_rankings', 'sqlite_sequence']


In [2]:
import requests

API_KEY = "P0lOvnPLuEIrxopGDL27YGRmvgHr9T2gMEUYGPvG"
BASE_URL = "https://api.sportradar.com/tennis/trial/v3/en"

# ---- LOAD Competitions + Categories ----
response = requests.get(
    f"{BASE_URL}/competitions.json?api_key={API_KEY}", timeout=10)
data = response.json()

categories_inserted = 0
competitions_inserted = 0

for comp in data['competitions']:
    # Insert category (ignore if already exists)
    cat = comp.get('category', {})
    if cat:
        cursor.execute("""
            INSERT OR IGNORE INTO categories (category_id, category_name)
            VALUES (?, ?)
        """, (cat.get('id'), cat.get('name')))
        categories_inserted += 1

    # Insert competition
    cursor.execute("""
        INSERT OR IGNORE INTO competitions 
        (competition_id, competition_name, parent_id, type, gender, category_id)
        VALUES (?, ?, ?, ?, ?, ?)
    """, (
        comp.get('id'),
        comp.get('name'),
        comp.get('parent_id'),
        comp.get('type', 'unknown'),
        comp.get('gender', 'unknown'),
        cat.get('id')
    ))
    competitions_inserted += 1

conn.commit()
print(f"✅ Categories inserted: {categories_inserted}")
print(f"✅ Competitions inserted: {competitions_inserted}")

✅ Categories inserted: 6572
✅ Competitions inserted: 6572


In [3]:
# ---- LOAD Complexes + Venues ----
response2 = requests.get(
    f"{BASE_URL}/complexes.json?api_key={API_KEY}", timeout=10)
data2 = response2.json()

complexes_inserted = 0
venues_inserted = 0

for complex_ in data2['complexes']:
    # Insert complex
    cursor.execute("""
        INSERT OR IGNORE INTO complexes (complex_id, complex_name)
        VALUES (?, ?)
    """, (complex_.get('id'), complex_.get('name')))
    complexes_inserted += 1

    # Insert each venue inside this complex
    for venue in complex_.get('venues', []):
        cursor.execute("""
            INSERT OR IGNORE INTO venues 
            (venue_id, venue_name, city_name, country_name, 
             country_code, timezone, complex_id)
            VALUES (?, ?, ?, ?, ?, ?, ?)
        """, (
            venue.get('id'),
            venue.get('name'),
            venue.get('city_name', 'Unknown'),
            venue.get('country_name', 'Unknown'),
            venue.get('country_code', 'UNK'),
            venue.get('timezone', 'Unknown'),
            complex_.get('id')
        ))
        venues_inserted += 1

conn.commit()
print(f"✅ Complexes inserted: {complexes_inserted}")
print(f"✅ Venues inserted:    {venues_inserted}")

✅ Complexes inserted: 764
✅ Venues inserted:    3975


In [4]:
# ---- LOAD Competitors + Rankings ----
response3 = requests.get(
    f"{BASE_URL}/double_competitors_rankings.json?api_key={API_KEY}",
    timeout=10)
data3 = response3.json()

competitors_inserted = 0
rankings_inserted = 0

for ranking_group in data3['rankings']:
    for entry in ranking_group.get('competitor_rankings', []):
        competitor = entry.get('competitor', {})

        # Insert competitor
        cursor.execute("""
            INSERT OR IGNORE INTO competitors
            (competitor_id, name, country, country_code, abbreviation)
            VALUES (?, ?, ?, ?, ?)
        """, (
            competitor.get('id'),
            competitor.get('name', 'Unknown'),
            competitor.get('country', 'Unknown'),
            competitor.get('country_code', 'UNK'),
            competitor.get('abbreviation', 'UNK')
        ))
        competitors_inserted += 1

        # Insert ranking
        cursor.execute("""
            INSERT OR IGNORE INTO competitor_rankings
            (rank, movement, points, competitions_played, competitor_id)
            VALUES (?, ?, ?, ?, ?)
        """, (
            entry.get('rank', 0),
            entry.get('movement', 0),
            entry.get('points', 0),
            entry.get('competitions_played', 0),
            competitor.get('id')
        ))
        rankings_inserted += 1

conn.commit()
print(f"✅ Competitors inserted: {competitors_inserted}")
print(f"✅ Rankings inserted:    {rankings_inserted}")

✅ Competitors inserted: 1000
✅ Rankings inserted:    1000


In [5]:
import pandas as pd

print("=" * 50)
print("COMPETITIONS QUERIES")
print("=" * 50)

# Query 1: List all competitions with their category name
print("\n--- Q1: All competitions with category name ---")
df1 = pd.read_sql_query("""
    SELECT c.competition_name, c.type, c.gender, cat.category_name
    FROM competitions c
    JOIN categories cat ON c.category_id = cat.category_id
    LIMIT 10
""", conn)
print(df1.to_string())

COMPETITIONS QUERIES

--- Q1: All competitions with category name ---
                    competition_name     type gender         category_name
0                         Hopman Cup    mixed  mixed            Hopman Cup
1                     World Team Cup    mixed    men                   ATP
2         ATP Challenger Tour Finals  singles    men            Challenger
3  Championship International Series  singles  women                   WTA
4                          Davis Cup    mixed    men             Davis Cup
5               Billie Jean King Cup    mixed  women  Billie Jean King Cup
6              Wimbledon Men Singles  singles    men                   ATP
7              Wimbledon Men Doubles  doubles    men                   ATP
8            Wimbledon Women Singles  singles  women                   WTA
9            Wimbledon Women Doubles  doubles  women                   WTA


In [6]:
# Query 2: Count competitions in each category
print("--- Q2: Count competitions per category ---")
df2 = pd.read_sql_query("""
    SELECT cat.category_name, COUNT(c.competition_id) as total_competitions
    FROM competitions c
    JOIN categories cat ON c.category_id = cat.category_id
    GROUP BY cat.category_name
    ORDER BY total_competitions DESC
    LIMIT 10
""", conn)
print(df2.to_string())

--- Q2: Count competitions per category ---
  category_name  total_competitions
0       ITF Men                2198
1     ITF Women                2032
2    Challenger                1021
3       UTR Men                 273
4           WTA                 259
5      WTA 125K                 242
6     UTR Women                 228
7           ATP                 225
8    Exhibition                  38
9   Wheelchairs                  16


In [7]:
# Query 3: All competitions of type 'doubles'
print("\n--- Q3: Competitions of type doubles ---")
df3 = pd.read_sql_query("""
    SELECT competition_name, gender, category_id
    FROM competitions
    WHERE type = 'doubles'
    LIMIT 10
""", conn)
print(df3.to_string())


--- Q3: Competitions of type doubles ---
                     competition_name gender    category_id
0               Wimbledon Men Doubles    men  sr:category:3
1             Wimbledon Women Doubles  women  sr:category:6
2         Australian Open Men Doubles    men  sr:category:3
3       Australian Open Women Doubles  women  sr:category:6
4             French Open Men Doubles    men  sr:category:3
5           French Open Women Doubles  women  sr:category:6
6                 US Open Men Doubles    men  sr:category:3
7               US Open Women Doubles  women  sr:category:6
8         ATP Doha, Qatar Men Doubles    men  sr:category:3
9  ATP Basel, Switzerland Men Doubles    men  sr:category:3


In [8]:
# Query 4: Competitions in ITF Men category
print("\n--- Q4: Competitions in ITF Men category ---")
df4 = pd.read_sql_query("""
    SELECT c.competition_name, c.type, c.gender
    FROM competitions c
    JOIN categories cat ON c.category_id = cat.category_id
    WHERE cat.category_name = 'ITF Men'
    LIMIT 10
""", conn)
print(df4.to_string())


--- Q4: Competitions in ITF Men category ---
                                   competition_name     type gender
0                 Dominican Republic F2 Men Singles  singles    men
1                 Dominican Republic F2 Men Doubles  doubles    men
2  ITF Men Stuttgart-Stammheim, Germany Men Singles  singles    men
3  ITF Men Stuttgart-Stammheim, Germany Men Doubles  doubles    men
4            ITF Men Cherkassy, Ukraine Men Singles  singles    men
5            ITF Men Cherkassy, Ukraine Men Doubles  doubles    men
6                      Great Britain F4 Men Singles  singles    men
7                      Great Britain F4 Men Doubles  doubles    men
8                ITF Men Vrsar, Croatia Men Singles  singles    men
9                ITF Men Vrsar, Croatia Men Doubles  doubles    men


In [9]:
# Query 5: Parent competitions and their sub-competitions
print("\n--- Q5: Parent and sub-competitions ---")
df5 = pd.read_sql_query("""
    SELECT p.competition_name AS parent_name,
           c.competition_name AS sub_competition
    FROM competitions c
    JOIN competitions p ON c.parent_id = p.competition_id
    LIMIT 10
""", conn)
print(df5.to_string())


--- Q5: Parent and sub-competitions ---
                   parent_name              sub_competition
0  ITF Romania F9, Men Singles  ITF Romania F9, Men Doubles


In [10]:
# Query 6: Distribution of competition types by category
print("\n--- Q6: Competition types distribution by category ---")
df6 = pd.read_sql_query("""
    SELECT cat.category_name, c.type, COUNT(*) as count
    FROM competitions c
    JOIN categories cat ON c.category_id = cat.category_id
    GROUP BY cat.category_name, c.type
    ORDER BY count DESC
    LIMIT 10
""", conn)
print(df6.to_string())


--- Q6: Competition types distribution by category ---
  category_name     type  count
0       ITF Men  doubles   1099
1       ITF Men  singles   1099
2     ITF Women  doubles   1018
3     ITF Women  singles   1014
4    Challenger  singles    511
5    Challenger  doubles    510
6       UTR Men  singles    273
7     UTR Women  singles    228
8           WTA  singles    130
9           WTA  doubles    129


In [11]:
# Query 7: Top-level competitions (no parent)
print("\n--- Q7: Top-level competitions (no parent) ---")
df7 = pd.read_sql_query("""
    SELECT competition_name, type, gender
    FROM competitions
    WHERE parent_id IS NULL
    LIMIT 10
""", conn)
print(df7.to_string())


--- Q7: Top-level competitions (no parent) ---
                           competition_name     type gender
0                                Hopman Cup    mixed  mixed
1                            World Team Cup    mixed    men
2         Championship International Series  singles  women
3                                 Davis Cup    mixed    men
4                      Billie Jean King Cup    mixed  women
5                                      IPTL  singles    men
6  ITF Men San Jose, Costa Rica Men Singles  singles    men
7  ITF Men San Jose, Costa Rica Men Doubles  doubles    men
8           ITF Colombia 02A, Women Singles  singles  women
9          ITF Australia 02B, Women Doubles  doubles  women


In [12]:
print("=" * 50)
print("VENUES & COMPLEXES QUERIES")
print("=" * 50)

# Query 1: All venues with complex name
print("\n--- Q1: Venues with complex name ---")
df_v1 = pd.read_sql_query("""
    SELECT v.venue_name, v.city_name, c.complex_name
    FROM venues v
    JOIN complexes c ON v.complex_id = c.complex_id
    LIMIT 10
""", conn)
print(df_v1.to_string())

VENUES & COMPLEXES QUERIES

--- Q1: Venues with complex name ---
       venue_name         city_name                complex_name
0  Cancha Central          Santiago                    Nacional
1    Centre Court           Seville          Estadio la Cartuja
2       Court One           Seville          Estadio la Cartuja
3     Sibur Arena  Saint Petersburg                 Sibur Arena
4    CENTER COURT  Saint Petersburg                 Sibur Arena
5       TC Dynamo  Saint Petersburg                 Sibur Arena
6         Court 1  Saint Petersburg                 Sibur Arena
7         Central            Oeiras  Complexo de Tenis do Jamor
8         Campo 1            Oeiras  Complexo de Tenis do Jamor
9         Campo 8            Oeiras  Complexo de Tenis do Jamor


In [13]:
# Query 2: Count venues in each complex
print("\n--- Q2: Venue count per complex ---")
df_v2 = pd.read_sql_query("""
    SELECT c.complex_name, COUNT(v.venue_id) as venue_count
    FROM complexes c
    JOIN venues v ON c.complex_id = v.complex_id
    GROUP BY c.complex_name
    ORDER BY venue_count DESC
    LIMIT 10
""", conn)
print(df_v2.to_string())


--- Q2: Venue count per complex ---
                             complex_name  venue_count
0                  National Tennis Center           75
1           Buenos Aires Lawn Tennis Club           29
2                          Melbourne Park           28
3                      Hurd Tennis Center           24
4                            Foro Italico           23
5  Club Tennis Las Terrazas de Miraflores           22
6                  Qi Zhong Tennis Center           21
7                Megasaray Tennis Academy           21
8              Indian Wells Tennis Garden           21
9                     Stade Roland Garros           20


In [14]:
# Query 3: Venues in Chile
print("\n--- Q3: Venues in Chile ---")
df_v3 = pd.read_sql_query("""
    SELECT venue_name, city_name, country_name, timezone
    FROM venues
    WHERE country_name = 'CHILE'
""", conn)
print(df_v3.to_string())


--- Q3: Venues in Chile ---
                 venue_name     city_name country_name          timezone
0            Cancha Central      Santiago        CHILE  America/Santiago
1                  Cancha 1        Temuco        CHILE  America/Santiago
2            Cancha Central        Temuco        CHILE  America/Santiago
3                  Cancha 3        Temuco        CHILE  America/Santiago
4                  Cancha 2        Temuco        CHILE  America/Santiago
5            Cancha Central  Vina del Mar        CHILE  America/Santiago
6                   Court 5  Vina del Mar        CHILE  America/Santiago
7                  Cancha 1  Vina del Mar        CHILE  America/Santiago
8                  Cancha 2  Vina del Mar        CHILE  America/Santiago
9        Court Jaime Fillol      Santiago        CHILE  America/Santiago
10  San Carlos De Apoquindo      Santiago        CHILE  America/Santiago
11           Cancha Central      Santiago        CHILE  America/Santiago
12                  Co

In [15]:
# Query 4: All venues and their timezones
print("\n--- Q4: Venues and timezones ---")
df_v4 = pd.read_sql_query("""
    SELECT venue_name, city_name, country_name, timezone
    FROM venues
    LIMIT 10
""", conn)
print(df_v4.to_string())


--- Q4: Venues and timezones ---
       venue_name         city_name        country_name          timezone
0  Cancha Central          Santiago               CHILE  America/Santiago
1    Centre Court           Seville               SPAIN     Europe/Madrid
2       Court One           Seville               SPAIN     Europe/Madrid
3     Sibur Arena  Saint Petersburg  RUSSIAN FEDERATION     Europe/Moscow
4    CENTER COURT  Saint Petersburg  RUSSIAN FEDERATION     Europe/Moscow
5       TC Dynamo  Saint Petersburg  RUSSIAN FEDERATION     Europe/Moscow
6         Court 1  Saint Petersburg  RUSSIAN FEDERATION     Europe/Moscow
7         Central            Oeiras            PORTUGAL     Europe/Lisbon
8         Campo 1            Oeiras            PORTUGAL     Europe/Lisbon
9         Campo 8            Oeiras            PORTUGAL     Europe/Lisbon


In [16]:
# Query 5: Complexes with more than one venue
print("\n--- Q5: Complexes with more than 1 venue ---")
df_v5 = pd.read_sql_query("""
    SELECT c.complex_name, COUNT(v.venue_id) as venue_count
    FROM complexes c
    JOIN venues v ON c.complex_id = v.complex_id
    GROUP BY c.complex_name
    HAVING venue_count > 1
    ORDER BY venue_count DESC
    LIMIT 10
""", conn)
print(df_v5.to_string())


--- Q5: Complexes with more than 1 venue ---
                             complex_name  venue_count
0                  National Tennis Center           75
1           Buenos Aires Lawn Tennis Club           29
2                          Melbourne Park           28
3                      Hurd Tennis Center           24
4                            Foro Italico           23
5  Club Tennis Las Terrazas de Miraflores           22
6                  Qi Zhong Tennis Center           21
7                Megasaray Tennis Academy           21
8              Indian Wells Tennis Garden           21
9                     Stade Roland Garros           20


In [17]:
# Query 6: Venues grouped by country
print("\n--- Q6: Venues grouped by country ---")
df_v6 = pd.read_sql_query("""
    SELECT country_name, COUNT(venue_id) as total_venues
    FROM venues
    GROUP BY country_name
    ORDER BY total_venues DESC
    LIMIT 10
""", conn)
print(df_v6.to_string())


--- Q6: Venues grouped by country ---
    country_name  total_venues
0  UNITED STATES           644
1          ITALY           293
2         FRANCE           276
3          CHINA           241
4          SPAIN           220
5        GERMANY           160
6      AUSTRALIA           139
7         MEXICO           138
8         BRAZIL           113
9        ENGLAND           112


In [18]:
# Query 7: Venues for Nacional complex
print("\n--- Q7: Venues for Nacional complex ---")
df_v7 = pd.read_sql_query("""
    SELECT v.venue_name, v.city_name, v.country_name
    FROM venues v
    JOIN complexes c ON v.complex_id = c.complex_id
    WHERE c.complex_name = 'Nacional'
""", conn)
print(df_v7.to_string())


--- Q7: Venues for Nacional complex ---
       venue_name city_name country_name
0  Cancha Central  Santiago        CHILE


In [21]:
print("=" * 50)
print("RANKINGS QUERIES")
print("=" * 50)

# Query 1: Top 10 ranked competitors
print("\n--- Q1: Top 10 ranked competitors ---")
df_r1 = pd.read_sql_query("""
    SELECT cr.rank, c.name, c.country, c.abbreviation, cr.points
    FROM competitor_rankings cr
    JOIN competitors c ON cr.competitor_id = c.competitor_id
    ORDER BY cr.rank ASC
    LIMIT 10
""", conn)
print(df_r1.to_string())

RANKINGS QUERIES

--- Q1: Top 10 ranked competitors ---
   rank                 name        country abbreviation  points
0     1    Heliovaara, Harri        Finland          HEL    8200
1     1        Patten, Henry  Great Britain          PAT    8200
2     1  Siniakova, Katerina        Czechia          SIN   10500
3     1    Heliovaara, Harri        Finland          HEL    8200
4     1        Patten, Henry  Great Britain          PAT    8200
5     1  Siniakova, Katerina        Czechia          SIN   10500
6     2     Townsend, Taylor            USA          TOW    9900
7     2     Townsend, Taylor            USA          TOW    9900
8     3    Zeballos, Horacio      Argentina          ZEB    8030
9     3  Dabrowski, Gabriela         Canada          DAB    7793


In [22]:
# Query 2: Competitors count by country
print("\n--- Q2: Competitors count by country ---")
df_r2 = pd.read_sql_query("""
    SELECT country, COUNT(*) as competitor_count
    FROM competitors
    GROUP BY country
    ORDER BY competitor_count DESC
    LIMIT 10
""", conn)
print(df_r2.to_string())


--- Q2: Competitors count by country ---
         country  competitor_count
0            USA               110
1  Great Britain                55
2         France                54
3        Neutral                49
4          Japan                48
5      Australia                45
6        Czechia                42
7          Italy                41
8      Argentina                39
9          Spain                34


In [23]:
# Query 3: Average points by country
print("\n--- Q3: Average ranking points by country ---")
df_r3 = pd.read_sql_query("""
    SELECT c.country, ROUND(AVG(cr.points), 2) as avg_points, COUNT(*) as player_count
    FROM competitors c
    JOIN competitor_rankings cr ON c.competitor_id = cr.competitor_id
    GROUP BY c.country
    HAVING player_count >= 3
    ORDER BY avg_points DESC
    LIMIT 10
""", conn)
print(df_r3.to_string())


--- Q3: Average ranking points by country ---
          country  avg_points  player_count
0         Finland     1889.00            10
1          Monaco     1772.33             6
2          Latvia     1475.50             8
3       Indonesia     1433.33             6
4      Kazakhstan     1387.67            12
5         Hungary     1259.83            12
6         Belgium     1146.18            22
7     New Zealand     1088.22            18
8  Chinese Taipei     1065.50            32
9         Croatia     1025.00            20


In [24]:
# Query 4: Competitors ranked in Top 100
print("\n--- Q4: Competitors ranked in Top 100 ---")
df_r4 = pd.read_sql_query("""
    SELECT cr.rank, c.name, c.country, cr.points, cr.competitions_played
    FROM competitor_rankings cr
    JOIN competitors c ON cr.competitor_id = c.competitor_id
    WHERE cr.rank <= 100
    ORDER BY cr.rank ASC
""", conn)
print(df_r4.to_string())


--- Q4: Competitors ranked in Top 100 ---
     rank                         name           country  points  competitions_played
0       1            Heliovaara, Harri           Finland    8200                   21
1       1                Patten, Henry     Great Britain    8200                   21
2       1          Siniakova, Katerina           Czechia   10500                   16
3       1            Heliovaara, Harri           Finland    8200                   21
4       1                Patten, Henry     Great Britain    8200                   21
5       1          Siniakova, Katerina           Czechia   10500                   16
6       2             Townsend, Taylor               USA    9900                   15
7       2             Townsend, Taylor               USA    9900                   15
8       3            Zeballos, Horacio         Argentina    8030                   14
9       3          Dabrowski, Gabriela            Canada    7793                   17
10      3  

In [25]:
# Query 5: Best ranked competitor per country
print("\n--- Q5: Best ranked competitor per country ---")
df_r5 = pd.read_sql_query("""
    SELECT c.country, c.name, cr.rank, cr.points
    FROM competitors c
    JOIN competitor_rankings cr ON c.competitor_id = cr.competitor_id
    WHERE cr.rank = (
        SELECT MIN(cr2.rank)
        FROM competitor_rankings cr2
        JOIN competitors c2 ON cr2.competitor_id = c2.competitor_id
        WHERE c2.country = c.country
    )
    ORDER BY cr.rank ASC
    LIMIT 15
""", conn)
print(df_r5.to_string())


--- Q5: Best ranked competitor per country ---
          country                 name  rank  points
0         Finland    Heliovaara, Harri     1    8200
1   Great Britain        Patten, Henry     1    8200
2         Czechia  Siniakova, Katerina     1   10500
3         Finland    Heliovaara, Harri     1    8200
4   Great Britain        Patten, Henry     1    8200
5         Czechia  Siniakova, Katerina     1   10500
6             USA     Townsend, Taylor     2    9900
7             USA     Townsend, Taylor     2    9900
8       Argentina    Zeballos, Horacio     3    8030
9          Canada  Dabrowski, Gabriela     3    7793
10      Argentina    Zeballos, Horacio     3    8030
11         Canada  Dabrowski, Gabriela     3    7793
12          Spain   Granollers, Marcel     4    7940
13        Belgium       Mertens, Elise     4    7298
14          Spain   Granollers, Marcel     4    7940


In [26]:
# Query 6: Competitors by points range
print("\n--- Q6: Competitors by points range ---")
df_r6 = pd.read_sql_query("""
    SELECT 
        CASE 
            WHEN points >= 5000 THEN '5000+'
            WHEN points >= 2000 THEN '2000-4999'
            WHEN points >= 1000 THEN '1000-1999'
            WHEN points >= 500  THEN '500-999'
            WHEN points >= 100  THEN '100-499'
            ELSE 'Below 100'
        END as points_range,
        COUNT(*) as competitor_count
    FROM competitor_rankings
    GROUP BY points_range
    ORDER BY MIN(points) DESC
""", conn)
print(df_r6.to_string())


--- Q6: Competitors by points range ---
  points_range  competitor_count
0        5000+                36
1    2000-4999               146
2    1000-1999               156
3      500-999               300
4      100-499              1362
